In [11]:
%pip install pandas sqlalchemy psycopg2-binary matplotlib seaborn
%pip install matplotlib seaborn pandas sqlalchemy psycopg2-binary

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine
from IPython.display import display, Markdown # ช่วยจัด format ใน Jupyter ให้สวย

# --- 1. Config Connection ---
DB_USER = 'warehouse_admin'
DB_PASS = 'warehouse_password'
DB_HOST = 'localhost'
DB_PORT = '5433'
DB_NAME = 'bitka_dw'

connection_str = f'postgresql://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(connection_str)

print("✅ Connecting to Data Warehouse...")

# ==========================================
# PART 1: OVERVIEW (ภาพรวมทั้งคลังข้อมูล)
# ==========================================
print("\n" + "="*50)
print("📊 DATA WAREHOUSE OVERVIEW")
print("="*50)

try:
    # ดึงรายชื่อตารางทั้งหมดใน Schema 'public'
    sql_tables = """
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public'
    """
    df_tables = pd.read_sql(sql_tables, engine)
    
    # วนลูปนับจำนวน Row ของแต่ละตาราง
    overview_data = []
    for table in df_tables['table_name']:
        count_df = pd.read_sql(f"SELECT COUNT(*) as cnt FROM {table}", engine)
        overview_data.append({'Table Name': table, 'Total Rows': count_df['cnt'].iloc[0]})
    
    df_overview = pd.DataFrame(overview_data).sort_values('Total Rows', ascending=False)
    display(df_overview)
    
except Exception as e:
    print(f"❌ Error fetching overview: {e}")

# ==========================================
# PART 2: AUTOMATED EXPLORATION (เจาะลึกทีละตาราง)
# ==========================================
print("\n" + "="*50)
print("🔍 DETAILED TABLE INSPECTION (Top 10 Rows & Stats)")
print("="*50)

for table in df_overview['Table Name']:
    display(Markdown(f"### 📁 Table: `{table}`"))
    
    # 2.1 ดึงข้อมูลตัวอย่าง 20 แถว
    df_sample = pd.read_sql(f"SELECT * FROM {table} ORDER BY 1 DESC LIMIT 10", engine)
    
    if not df_sample.empty:
        # แปลงข้อมูลเวลาให้เป็น Datetime อัตโนมัติ (DS ชอบสิ่งนี้)
        cols = df_sample.columns
        for c in cols:
            if 'time' in c or 'date' in c or '_at' in c:
                try:
                    df_sample[c] = pd.to_datetime(df_sample[c])
                except:
                    pass

        # 2.2 โชว์ข้อมูลดิบ
        print(f"🔹 Sample Data (20 rows):")
        display(df_sample)
        
        # 2.3 โชว์ Statistics (Numerical Distribution)
        # สิ่งนี้ช่วยให้เห็น Mean, Min, Max, SD ทันที
        print(f"🔹 Descriptive Statistics:")
        display(df_sample.describe(include='all'))
        
        print("-" * 80)
    else:
        print("⚠️ Table is empty.")
        print("-" * 80)

# ==========================================
# PART 3: ANALYSIS EXAMPLE (ตัวอย่างการวิเคราะห์เจาะจง)
# ==========================================
display(Markdown("## 📈 Example Analysis: BTC_THB Price & Volume"))

# Example 1: Price Trend
sql_price = """
SELECT updated_at, last_price 
FROM tickers 
WHERE symbol = 'BTC_THB' 
ORDER BY updated_at ASC 
"""
df_price = pd.read_sql(sql_price, engine)

if not df_price.empty:
    df_price['last_price'] = df_price['last_price'].astype(float)
    df_price['updated_at'] = pd.to_datetime(df_price['updated_at'])

    plt.figure(figsize=(10, 4))
    sns.lineplot(data=df_price, x='updated_at', y='last_price', marker='o', color='orange')
    plt.title('BTC_THB Price Trend')
    plt.xticks(rotation=45)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Example 2: Order Side Distribution
sql_orders = """
SELECT side, count(*) as count 
FROM orders 
WHERE symbol = 'BTC_THB' 
GROUP BY side
"""
df_orders = pd.read_sql(sql_orders, engine)

if not df_orders.empty:
    plt.figure(figsize=(4, 4))
    plt.pie(df_orders['count'], labels=df_orders['side'], autopct='%1.1f%%', colors=['#66b3ff', '#ff9999'])
    plt.title('BTC_THB Buy vs Sell Ratio')
    plt.show()


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python3.13 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
✅ Connecting to Data Warehouse...

📊 DATA WAREHOUSE OVERVIEW


,Table Name,Total Rows
5,matches,21000
4,users,22
0,deposits,0
1,login_history,0
2,audit_logs,0
3,withdrawals,0
6,orders,0
7,tickers,0



🔍 DETAILED TABLE INSPECTION (Top 10 Rows & Stats)


### 📁 Table: `matches`

🔹 Sample Data (20 rows):


...

### 📁 Table: `orders`

⚠️ Table is empty.
--------------------------------------------------------------------------------


### 📁 Table: `tickers`

⚠️ Table is empty.
--------------------------------------------------------------------------------


## 📈 Example Analysis: BTC_THB Price & Volume